# Survey: All h11=3 Calabi-Yau threefolds

This notebook loads JSONL results from `tests/survey_h11_3.py` and computes
statistics on phases, contraction types, Coxeter groups, cone generators, and timing.

Works with partial results (e.g., after `--limit 10`) or the full survey.

In [ ]:
import json
import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

# Load JSONL results
jsonl_path = os.path.join("..", "tests", "survey_h11_3_results.jsonl")
all_results = []
with open(jsonl_path) as f:
    for line in f:
        line = line.strip()
        if line:
            all_results.append(json.loads(line))

# Split into ok and error
results = [r for r in all_results if r["status"] == "ok"]
errors = [r for r in all_results if r["status"] == "error"]

print(f"Loaded {len(all_results)} results ({len(results)} ok, {len(errors)} errors)")

## Phase count distribution

In [ ]:
fund_phases = [r["n_phases_fund"] for r in results]
total_phases = [r["n_phases_total"] for r in results]

print(f"Fundamental domain phases:")
print(f"  mean={np.mean(fund_phases):.1f}  median={np.median(fund_phases):.0f}  max={np.max(fund_phases)}")
print(f"Total phases (after orbit):")
print(f"  mean={np.mean(total_phases):.1f}  median={np.median(total_phases):.0f}  max={np.max(total_phases)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(fund_phases, bins=range(1, max(fund_phases) + 2), align="left", edgecolor="black")
axes[0].set_xlabel("Number of phases")
axes[0].set_ylabel("Count")
axes[0].set_title("Fundamental domain phases")

axes[1].hist(total_phases, bins=range(1, max(total_phases) + 2), align="left", edgecolor="black")
axes[1].set_xlabel("Number of phases")
axes[1].set_ylabel("Count")
axes[1].set_title("Total phases (after orbit expansion)")

plt.tight_layout()
plt.show()

## Contraction type distribution

In [ ]:
# Aggregate contraction types across all polytopes
total_types = Counter()
polytopes_with_type = Counter()  # how many polytopes have at least one of each type

for r in results:
    ct = r["contraction_types"]
    total_types.update(ct)
    for t in ct:
        polytopes_with_type[t] += 1

print("Total contraction counts across all polytopes:")
for name, count in total_types.most_common():
    print(f"  {name}: {count}")

print(f"\nPolytopes with at least one of each type:")
for name, count in polytopes_with_type.most_common():
    print(f"  {name}: {count}/{len(results)}")

# Bar chart
names = [n for n, _ in total_types.most_common()]
counts = [c for _, c in total_types.most_common()]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(names, counts, edgecolor="black")
ax.set_ylabel("Total count")
ax.set_title("Contraction types across all polytopes")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Coxeter group summary

In [ ]:
# Coxeter type distribution
cox_counter = Counter()
for r in results:
    ct = r["coxeter_type"]
    key = str(ct) if ct else "trivial"
    cox_counter[key] += 1

n_nontrivial = sum(1 for r in results if r["coxeter_type"] is not None)
print(f"Polytopes with nontrivial Coxeter group: {n_nontrivial}/{len(results)}")
print()

print(f"{'Coxeter type':<30} {'Order':>8} {'Count':>6}")
print("-" * 50)
for key, count in cox_counter.most_common():
    # Find an example to get the order
    if key == "trivial":
        order_str = "1"
    else:
        example = next(r for r in results if str(r["coxeter_type"]) == key)
        order_str = str(example["coxeter_order"])
    print(f"{key:<30} {order_str:>8} {count:>6}")

# Order distribution for nontrivial
orders = [r["coxeter_order"] for r in results if r["coxeter_order"] is not None]
if orders:
    print(f"\nCoxeter group orders (nontrivial only):")
    order_counter = Counter(orders)
    for o, c in sorted(order_counter.items()):
        finite_str = "finite" if o < float("inf") else "infinite"
        print(f"  order={o} ({finite_str}): {c} polytopes")

## Cone generator counts

In [ ]:
inf_gens = [r["n_infinity_gens"] for r in results]
eff_gens = [r["n_eff_gens"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(inf_gens, bins=range(0, max(inf_gens) + 2), align="left", edgecolor="black")
axes[0].set_xlabel("Number of generators")
axes[0].set_ylabel("Count")
axes[0].set_title("Infinity cone generators")

axes[1].hist(eff_gens, bins=range(0, max(eff_gens) + 2), align="left", edgecolor="black")
axes[1].set_xlabel("Number of generators")
axes[1].set_ylabel("Count")
axes[1].set_title("Effective cone generators")

plt.tight_layout()
plt.show()

## Timing

In [ ]:
times = [r["time_s"] for r in results]

print(f"Timing statistics:")
print(f"  mean={np.mean(times):.1f}s  median={np.median(times):.1f}s  total={np.sum(times):.0f}s")
print(f"  min={np.min(times):.1f}s  max={np.max(times):.1f}s")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(times, bins=30, edgecolor="black")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Count")
ax.set_title("Per-polytope computation time")
plt.tight_layout()
plt.show()

## Errors and unresolved walls

In [ ]:
# Errors
if errors:
    print(f"Polytopes with errors ({len(errors)}):")
    for r in errors:
        print(f"  #{r['poly_id']}: {r['error']}")
else:
    print("No errors.")

# Unresolved walls
unresolved = [r for r in results if r["n_unresolved"] > 0]
if unresolved:
    print(f"\nPolytopes with unresolved walls ({len(unresolved)}):")
    for r in unresolved:
        print(f"  #{r['poly_id']}: {r['n_unresolved']} unresolved")
else:
    print("No unresolved walls.")

## Summary table

In [ ]:
# Sort by n_phases_total descending
sorted_results = sorted(results, key=lambda r: r["n_phases_total"], reverse=True)

print(f"{'poly_id':>7}  {'fund':>4}  {'total':>5}  {'coxeter_type':<25}  {'order':>6}  {'time_s':>6}")
print("-" * 65)
for r in sorted_results:
    cox_str = str(r['coxeter_type']) if r['coxeter_type'] else "-"
    order_str = str(r['coxeter_order']) if r['coxeter_order'] else "-"
    print(f"{r['poly_id']:7d}  {r['n_phases_fund']:4d}  {r['n_phases_total']:5d}  "
          f"{cox_str:<25}  {order_str:>6}  {r['time_s']:6.1f}")